# Copy Cars from PRD to NPR

This notebook:
1. Deletes all cars from NPR Firestore
2. Deletes all car-related files from NPR Storage
3. Copies all cars from PRD Firestore to NPR
4. Copies all car-related files from PRD Storage to NPR

**WARNING**: This will permanently delete all car data in NPR before copying from PRD!

In [1]:
from google.cloud import firestore
from google.cloud import storage

In [6]:
# Configuration
SOURCE_PROJECT_ID = "wilde-rosen-483d1"
DEST_PROJECT_ID = "wilde-rosen-npr-dfafc"

# Storage bucket names
SOURCE_BUCKET_NAME = f"{SOURCE_PROJECT_ID}.firebasestorage.app"
DEST_BUCKET_NAME = f"{DEST_PROJECT_ID}.firebasestorage.app"

print(f"Source (PRD): {SOURCE_PROJECT_ID}")
print(f"Destination (NPR): {DEST_PROJECT_ID}")
print(f"Source Bucket: {SOURCE_BUCKET_NAME}")
print(f"Destination Bucket: {DEST_BUCKET_NAME}")

Source (PRD): wilde-rosen-483d1
Destination (NPR): wilde-rosen-npr-dfafc
Source Bucket: wilde-rosen-483d1.firebasestorage.app
Destination Bucket: wilde-rosen-npr-dfafc.firebasestorage.app


In [8]:
# Initialize Firestore clients
source_db = firestore.Client(project=SOURCE_PROJECT_ID)
dest_db = firestore.Client(project=DEST_PROJECT_ID)

# Initialize Storage clients
storage_client = storage.Client()
source_bucket = storage_client.bucket(SOURCE_BUCKET_NAME)
dest_bucket = storage_client.bucket(DEST_BUCKET_NAME)

print("✓ Clients initialized")

✓ Clients initialized


## Step 1: Delete all cars from NPR Firestore

This will delete all documents in the `cars` collection and their subcollections.

In [4]:
def delete_damages(db):
    """Delete all damages from all cars."""
    cars = db.collection('cars').stream()
    for car in cars:
        damages = car.reference.collection('damages').stream()
        for damage in damages:
            damage.reference.delete()
        print(f"  Deleted damages for car: {car.id}")

print("Deleting all damages from NPR Firestore...")
delete_damages(dest_db)
print("✓ Deleted all damages from NPR Firestore")

Deleting all damages from NPR Firestore...
  Deleted damages for car: jogger
  Deleted damages for car: kangoo
  Deleted damages for car: kona
  Deleted damages for car: zoe
✓ Deleted all damages from NPR Firestore


## Step 2: Delete all car files from NPR Storage

This will delete all files in the `cars/` folder in Storage.

In [9]:
def delete_storage_folder(bucket, prefix: str):
    """
    Delete all blobs with the given prefix.
    """
    blobs = list(bucket.list_blobs(prefix=prefix))
    deleted = 0
    
    for blob in blobs:
        blob.delete()
        deleted += 1
        if deleted % 10 == 0:
            print(f"  Deleted {deleted} files...")
    
    return deleted

print("Deleting all car files from NPR Storage...")
deleted_files = delete_storage_folder(dest_bucket, 'cars/')
print(f"✓ Deleted {deleted_files} files from NPR Storage")

Deleting all car files from NPR Storage...
  Deleted 10 files...
  Deleted 20 files...
  Deleted 30 files...
  Deleted 40 files...
✓ Deleted 40 files from NPR Storage


## Step 3: Copy all cars from PRD to NPR Firestore

This will copy all car documents and their subcollections.

In [10]:
def copy_document_with_subcollections(source_ref, dest_ref):
    """
    Recursively copy a document and all its subcollections.
    """
    # Get the source document
    source_doc = source_ref.get()
    
    if not source_doc.exists:
        return 0
    
    # Copy the document data
    dest_ref.set(source_doc.to_dict())
    copied = 1
    
    # Copy all subcollections
    for subcollection in source_ref.collections():
        subcollection_name = subcollection.id
        
        # Get all documents in the subcollection
        for subdoc in subcollection.stream():
            source_subdoc_ref = source_ref.collection(subcollection_name).document(subdoc.id)
            dest_subdoc_ref = dest_ref.collection(subcollection_name).document(subdoc.id)
            copied += copy_document_with_subcollections(source_subdoc_ref, dest_subdoc_ref)
    
    return copied

print("Copying all cars from PRD to NPR Firestore...")
cars_ref = source_db.collection('cars')
total_copied = 0

for car_doc in cars_ref.stream():
    car_id = car_doc.id
    print(f"Copying car: {car_id}")
    
    source_car_ref = source_db.collection('cars').document(car_id)
    dest_car_ref = dest_db.collection('cars').document(car_id)
    
    copied = copy_document_with_subcollections(source_car_ref, dest_car_ref)
    total_copied += copied
    print(f"  ✓ Copied {copied} documents for {car_id}")

print(f"\n✓ Total documents copied: {total_copied}")

Copying all cars from PRD to NPR Firestore...
Copying car: jogger
  ✓ Copied 1 documents for jogger
Copying car: kangoo
  ✓ Copied 5 documents for kangoo
Copying car: kona
  ✓ Copied 10 documents for kona
Copying car: zoe
  ✓ Copied 14 documents for zoe

✓ Total documents copied: 30


## Step 4: Copy all car files from PRD to NPR Storage

This will copy all files in the `cars/` folder.

In [11]:
def copy_storage_folder(source_bucket, dest_bucket, prefix: str):
    """
    Copy all blobs with the given prefix from source to destination bucket.
    """
    blobs = list(source_bucket.list_blobs(prefix=prefix))
    copied = 0
    
    for blob in blobs:
        # Copy blob to destination bucket with same name
        source_bucket.copy_blob(
            blob, dest_bucket, blob.name
        )
        copied += 1
        
        if copied % 10 == 0:
            print(f"  Copied {copied} files...")
    
    return copied

print("Copying all car files from PRD to NPR Storage...")
copied_files = copy_storage_folder(source_bucket, dest_bucket, 'cars/')
print(f"✓ Copied {copied_files} files to NPR Storage")

Copying all car files from PRD to NPR Storage...
  Copied 10 files...
  Copied 20 files...
  Copied 30 files...
  Copied 40 files...
  Copied 50 files...
  Copied 60 files...
  Copied 70 files...
✓ Copied 71 files to NPR Storage


## Step 5: Verification

Verify that the data was copied correctly.

In [13]:
# Count total damages in both databases
source_damage_count = 0
for car in source_db.collection('cars').stream():
    damages = list(car.reference.collection('damages').stream())
    source_damage_count += len(damages)

dest_damage_count = 0
for car in dest_db.collection('cars').stream():
    damages = list(car.reference.collection('damages').stream())
    dest_damage_count += len(damages)

print(f"Total damages in PRD: {source_damage_count}")
print(f"Total damages in NPR: {dest_damage_count}")

# Count files in both buckets
source_files = list(source_bucket.list_blobs(prefix='cars/'))
dest_files = list(dest_bucket.list_blobs(prefix='cars/'))

print(f"\nFiles in PRD Storage: {len(source_files)}")
print(f"Files in NPR Storage: {len(dest_files)}")

if source_damage_count == dest_damage_count and len(source_files) == len(dest_files):
    print("\n✓ Verification successful! All data copied.")
else:
    print("\n⚠ Warning: Counts don't match. Please review.")

Total damages in PRD: 26
Total damages in NPR: 26

Files in PRD Storage: 71
Files in NPR Storage: 71

✓ Verification successful! All data copied.


## Summary

Run all cells above in sequence to:
1. Clean NPR environment
2. Copy all car data from PRD to NPR
3. Verify the copy was successful